# LegalQA dev100 — full retrieval, reuse Version 3

Notebook này chạy đúng 100 câu `dev100` bằng cấu hình retrieval đầy đủ. Full index 407.107 chunks và model weights được đọc trực tiếp từ output Kaggle Version 3 (`scriptVersionId=348583427`), không build embeddings lại.

Trước khi Save & Run All, trong **Add Input → Datasets**, gắn hai dataset `lighth/ver3-smoke-output` và `lighth/uit-dsc-2026-task2-legalqa-train`.

## 1. Cấu hình đường dẫn và run

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

if not Path('/kaggle').exists():
    raise RuntimeError('Notebook dev100 này chỉ được cấu hình để chạy trên Kaggle.')

VERSION3_URL = 'https://www.kaggle.com/datasets/lighth/ver3-smoke-output'
VERSION3_ROOT = Path('/kaggle/input/datasets/lighth/ver3-smoke-output/legalqa_smoke_full_v1')
SAVED_INDEX = VERSION3_ROOT / 'index'
SAVED_MODELS = VERSION3_ROOT / 'models'

REPO_URL = 'https://github.com/lighth-gh/uit-dsc-2026-task2-legalqa.git'
REPO_REF = 'main'
WORK_BASE = Path('/kaggle/working')
CODE = WORK_BASE / 'uit-dsc-2026-task2-legalqa'
RUN_NAME = 'legalqa_dev100_full_retrieval_v5'
RUN_ROOT = WORK_BASE / RUN_NAME

USE_REPO_DATA = False
KAGGLE_DATASET_ROOT = Path('/kaggle/input/datasets/lighth/uit-dsc-2026-task2-legalqa-train')
DEV_SPLIT = 'dev100'
EXPECTED_QUESTIONS = 100
GENERATION_MODE = 'generate'

MODELS = RUN_ROOT / 'models'  # Lock/audit và symlink tới weights Version 3.
INDEX = SAVED_INDEX           # Đọc trực tiếp full index; không copy, không rebuild.
DATA = RUN_ROOT / 'data'
CFG = RUN_ROOT / 'dev100_config.json'
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print('Version 3 input:', VERSION3_ROOT)
print('Dev100 output:', RUN_ROOT)

## 2. Clone code, kiểm tra Version 3 và khóa cấu hình

Cell này dừng ngay nếu input Version 3 thiếu index/model. Retrieval dùng cấu hình đầy đủ; generation giữ 1024 output tokens để so sánh nhất quán với smoke Version 4.

In [ ]:
if CODE.exists():
    if not (CODE / '.git').is_dir():
        raise RuntimeError(f'{CODE} đã tồn tại nhưng không phải Git repo. Hãy Restart Session hoặc đổi CODE.')
    remote = subprocess.check_output(['git', '-C', str(CODE), 'remote', 'get-url', 'origin'], text=True).strip()
    if remote.rstrip('/') != REPO_URL.rstrip('/'):
        raise RuntimeError(f'Remote không đúng repo yêu cầu: {remote}')
    subprocess.run(['git', '-C', str(CODE), 'pull', '--ff-only', 'origin', REPO_REF], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(CODE)], check=True)

required_index = ['index_manifest.json', 'corpus.sqlite', 'dense.faiss']
missing_index = [name for name in required_index if not (SAVED_INDEX / name).is_file()]
required_models = ['models.lock.json']
missing_models = [name for name in required_models if not (SAVED_MODELS / name).is_file()]
missing_roles = [role for role in ['embedding', 'reranker', 'generator'] if not (SAVED_MODELS / role / 'config.json').is_file()]
if missing_index or missing_models or missing_roles:
    raise FileNotFoundError(
        'Thiếu artifacts Version 3. Hãy Add Input dataset lighth/ver3-smoke-output từ ' + VERSION3_URL
        + f' | index={missing_index}, model_files={missing_models}, model_roles={missing_roles}'
    )

MODELS.mkdir(parents=True, exist_ok=True)
for role in ['embedding', 'reranker', 'generator']:
    source_dir = SAVED_MODELS / role
    link = MODELS / role
    if link.exists() or link.is_symlink():
        if link.resolve() != source_dir.resolve():
            raise RuntimeError(f'Symlink model không đúng: {link} -> {link.resolve()}')
    else:
        link.symlink_to(source_dir, target_is_directory=True)
shutil.copy2(SAVED_MODELS / 'models.lock.json', MODELS / 'models.lock.json')

DATASET_ROOT = CODE if USE_REPO_DATA else KAGGLE_DATASET_ROOT
TRAIN_PATH = DATASET_ROOT / 'train.json'
TEST_PATH = DATASET_ROOT / 'public-official.json'
for label, data_path in [('train', TRAIN_PATH), ('test', TEST_PATH)]:
    if not data_path.is_file():
        raise FileNotFoundError(f'{label}: {data_path}')

dev_cfg = json.loads((CODE / 'config.json').read_text(encoding='utf-8'))
dev_cfg['retrieval'].update({
    'bm25_k': 100,
    'dense_k': 100,
    'rrf_constant': 60,
    'pool_k': 24,
    'max_children_per_parent': 2,
    'parents_k': 4,
    'embedding_batch': 32,
    'reranker_batch': 8,
    'reranker_max_tokens': 768,
})
dev_cfg['generation'].update({
    'max_input_tokens': 4096,
    'max_new_tokens': 1024,
    'parent_max_tokens': 1400,
    'min_context_tokens': 256,
})
CFG.write_text(json.dumps(dev_cfg, ensure_ascii=False, indent=2), encoding='utf-8')

def run(*args):
    command = [sys.executable, '-m', 'legalqa', '--config', str(CFG), '--models', str(MODELS), *map(str, args)]
    print('Running:', ' '.join(command), flush=True)
    subprocess.run(command, cwd=CODE, check=True)

commit = subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Code:', CODE, '| Commit:', commit)
print('Saved index:', INDEX)
print('Model links:', {role: str((MODELS / role).resolve()) for role in ['embedding', 'reranker', 'generator']})

## 3. Cài dependencies và chạy test

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'nltk.downloader', '-q', 'wordnet', 'omw-1.4'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=CODE, check=True)
subprocess.run([sys.executable, 'scripts/check_metrics.py'], cwd=CODE, check=True)
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(RUN_ROOT / 'environment.freeze.txt').write_text(freeze, encoding='utf-8')

## 4. Chuẩn bị đúng split dev100

In [ ]:
run('prepare', '--train', TRAIN_PATH, '--test', TEST_PATH, '--output', DATA)
report = json.loads((DATA / 'data_report.json').read_text(encoding='utf-8'))
assert report['split_sizes']['dev100'] == EXPECTED_QUESTIONS, report['split_sizes']
print(json.dumps(report, ensure_ascii=False, indent=2))
print('Dev100 data prepared successfully.')

## 5. Audit model và xác nhận full index Version 3

In [ ]:
run('audit-models')
audit = json.loads((MODELS / 'parameter_audit.json').read_text(encoding='utf-8'))
assert audit['passes'] and audit['total_with_unmerged_lora'] < 4_000_000_000
print('Parameter audit OK:', f"{audit['total_with_unmerged_lora']:,}")

manifest = json.loads((INDEX / 'index_manifest.json').read_text(encoding='utf-8'))
if manifest.get('chunks') != 407_107:
    raise RuntimeError(f"Không phải full index Version 3: chunks={manifest.get('chunks')}")
if manifest.get('documents') != 8_507:
    raise RuntimeError(f"Số documents không khớp Version 3: documents={manifest.get('documents')}")
print('Reusing completed Version 3 index:', INDEX)
print(json.dumps(manifest, ensure_ascii=False, indent=2))

## 6. Retrieve, generate và evaluate 100 câu

Nếu session bị ngắt nhưng chưa bị reset, chạy lại cell này sẽ tiếp tục từ checkpoint có cùng fingerprint.

In [ ]:
from collections import Counter

DEV100_QUESTIONS_PATH = DATA / f'{DEV_SPLIT}.questions.json'
DEV100_REFERENCES_PATH = DATA / f'{DEV_SPLIT}.references.json'
DEV100_RETRIEVAL = RUN_ROOT / 'dev100.retrieval.json'
DEV100_PREDICTIONS = RUN_ROOT / 'dev100.predictions.json'
DEV100_METRICS = RUN_ROOT / 'dev100.metrics.json'

dev_questions = json.loads(DEV100_QUESTIONS_PATH.read_text(encoding='utf-8'))
dev_references = json.loads(DEV100_REFERENCES_PATH.read_text(encoding='utf-8'))
assert len(dev_questions) == EXPECTED_QUESTIONS, f'Expected {EXPECTED_QUESTIONS}, found {len(dev_questions)}'
assert set(dev_questions) == set(dev_references), 'Question IDs and reference IDs do not match.'

print(f'Running retrieval for {len(dev_questions)} dev100 questions...')
run('retrieve', '--questions', DEV100_QUESTIONS_PATH, '--index', INDEX, '--output', DEV100_RETRIEVAL)

print('Retrieval completed. Starting generation...')
run('generate', '--questions', DEV100_QUESTIONS_PATH, '--retrieval', DEV100_RETRIEVAL, '--output', DEV100_PREDICTIONS, '--mode', GENERATION_MODE)

print('Generation completed. Starting evaluation...')
run('evaluate', '--predictions', DEV100_PREDICTIONS, '--references', DEV100_REFERENCES_PATH, '--output', DEV100_METRICS, '--label', 'dev100_full_retrieval')

predictions = json.loads(DEV100_PREDICTIONS.read_text(encoding='utf-8'))
metrics = json.loads(DEV100_METRICS.read_text(encoding='utf-8'))
generation_audit = json.loads(DEV100_PREDICTIONS.with_suffix('.audit.json').read_text(encoding='utf-8'))
assert len(predictions) == EXPECTED_QUESTIONS
assert metrics['samples'] == EXPECTED_QUESTIONS

routes = Counter(item['route'] for item in generation_audit.values())
token_limit_count = sum(bool(item['hit_token_limit']) for item in generation_audit.values())
summary = {
    'samples': metrics['samples'],
    'meteor': metrics['meteor'],
    'rougeL': metrics['rougeL'],
    'routes': dict(routes),
    'token_limit_count': token_limit_count,
    'token_limit_rate': metrics.get('token_limit_rate'),
}
print('\nDEV100 PASSED')
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('\nOutput directory:', RUN_ROOT)

for key in list(dev_questions)[:3]:
    print('\nID:', key)
    print('Question:', dev_questions[key]['question'])
    print('Prediction:', predictions[key]['answer'])
    print('Reference:', dev_references[key])

## 7. Đóng gói diagnostics

ZIP chỉ chứa cấu hình và kết quả dev100; không chứa model weights hoặc full index.

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

DIAGNOSTICS_ZIP = RUN_ROOT / 'legalqa_dev100_diagnostics.zip'
diagnostic_files = [
    CFG,
    RUN_ROOT / 'environment.freeze.txt',
    DATA / 'data_report.json',
    DATA / 'split_manifest.json',
    DEV100_QUESTIONS_PATH,
    DEV100_REFERENCES_PATH,
    DEV100_RETRIEVAL,
    DEV100_PREDICTIONS,
    DEV100_PREDICTIONS.with_suffix('.audit.json'),
    DEV100_PREDICTIONS.with_suffix('.manifest.json'),
    DEV100_METRICS,
]

with ZipFile(DIAGNOSTICS_ZIP, 'w', compression=ZIP_DEFLATED) as archive:
    for file_path in diagnostic_files:
        if file_path.is_file():
            archive.write(file_path, arcname=file_path.name)
        else:
            print('Warning: missing', file_path)

print('Diagnostics ZIP:', DIAGNOSTICS_ZIP)
print('Size MB:', round(DIAGNOSTICS_ZIP.stat().st_size / 1024**2, 2))

## Điều kiện hoàn thành

Run đạt khi log có `Reusing completed Version 3 index`, `Retrieved: 100/100`, `Answered: 100/100` và `DEV100 PASSED`. Output được lưu tại `/kaggle/working/legalqa_dev100_full_retrieval_v5`.